In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.utils import degree
from sklearn.metrics import roc_auc_score, average_precision_score
import random
import pickle
import numpy as np
from torch_geometric.transforms import ToUndirected, AddSelfLoops
from torch_geometric.data import HeteroData
import torch.nn as nn
import copy
from tqdm import tqdm




c:\Users\tirth\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

with open('bio_graph_raw.pkl', 'rb') as f:
    G = pickle.load(f)
# Separate node types
chemicals = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'chemical'])
diseases  = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'disease'])
genes     = sorted([n for n, d in G.nodes(data=True) if d['node_type'] == 'gene'])

chem_idx = {c: i for i, c in enumerate(chemicals)}
disease_idx = {d: i for i, d in enumerate(diseases)}
gene_idx = {g: i for i, g in enumerate(genes)}

print(chem_idx)

# Raw features (NO padding)
chem_features = torch.tensor(
    np.array([G.nodes[c]['x'] for c in chemicals], dtype=np.float32)
)

disease_features = torch.tensor(
    np.array([G.nodes[d]['x'] for d in diseases], dtype=np.float32)
)

gene_features = torch.tensor(
    np.array([G.nodes[g]['x'] for g in genes], dtype=np.float32)
)

# Edge extraction
chem_gene_edges = []
chem_disease_edges = []
gene_disease_edges = []

for u, v, data in G.edges(data=True):
    if data['edge_type'] == 'chem_gene':
        chem_gene_edges.append([chem_idx[u], gene_idx[v]])
    elif data['edge_type'] == 'chem_disease':
        chem_disease_edges.append([chem_idx[u], disease_idx[v]])
    elif data['edge_type'] == 'gene_disease':
        gene_disease_edges.append([gene_idx[u], disease_idx[v]])

chem_gene_edges = torch.tensor(chem_gene_edges).t().contiguous()
chem_disease_edges = torch.tensor(chem_disease_edges).t().contiguous()
gene_disease_edges = torch.tensor(gene_disease_edges).t().contiguous()

# Build HeteroData
data = HeteroData()

data['chemical'].x = chem_features
data['disease'].x  = disease_features
data['gene'].x     = gene_features

data['chemical','chem_gene','gene'].edge_index = chem_gene_edges
data['chemical','chem_disease','disease'].edge_index = chem_disease_edges
data['gene','gene_disease','disease'].edge_index = gene_disease_edges

data = ToUndirected()(data)
data = AddSelfLoops()(data)

print(data)

{'C000015': 0, 'C000025': 1, 'C000050': 2, 'C000061': 3, 'C000081': 4, 'C000090': 5, 'C000109': 6, 'C000121': 7, 'C000123': 8, 'C000152': 9, 'C000154': 10, 'C000188': 11, 'C000228': 12, 'C000297': 13, 'C000298': 14, 'C000380': 15, 'C000433': 16, 'C000449': 17, 'C000470': 18, 'C000474': 19, 'C000475': 20, 'C000481': 21, 'C000482': 22, 'C000483': 23, 'C000488': 24, 'C000490': 25, 'C000499': 26, 'C000505': 27, 'C000511': 28, 'C000515': 29, 'C000516': 30, 'C000518': 31, 'C000520': 32, 'C000536': 33, 'C000541': 34, 'C000543': 35, 'C000588486': 36, 'C000588503': 37, 'C000588556': 38, 'C000588664': 39, 'C000588695': 40, 'C000588741': 41, 'C000588809': 42, 'C000588918': 43, 'C000589002': 44, 'C000589029': 45, 'C000589147': 46, 'C000589162': 47, 'C000589253': 48, 'C000589451': 49, 'C000589472': 50, 'C000589490': 51, 'C000589596': 52, 'C000589651': 53, 'C000589733': 54, 'C000589878': 55, 'C000589977': 56, 'C000590225': 57, 'C000590451': 58, 'C000590645': 59, 'C000590659': 60, 'C000590786': 61, '

In [3]:
edge_type = ('chemical','chem_disease','disease')

edge_index = data[edge_type].edge_index
num_edges = edge_index.size(1)

perm = torch.randperm(num_edges)

train_size = int(0.8 * num_edges)
val_size = int(0.1 * num_edges)

train_idx = perm[:train_size]
val_idx   = perm[train_size:train_size+val_size]
test_idx  = perm[train_size+val_size:]

train_pos = edge_index[:, train_idx]
val_pos   = edge_index[:, val_idx]
test_pos  = edge_index[:, test_idx]


data[edge_type].edge_index = train_pos


In [4]:
num_chem = data['chemical'].num_nodes
num_dis  = data['disease'].num_nodes

chem_deg = degree(train_pos[0], num_chem)
dis_deg  = degree(train_pos[1], num_dis)

def create_bins(deg, num_bins=10):
    log_deg = torch.log1p(deg)
    bins = torch.linspace(log_deg.min(), log_deg.max(), num_bins + 1)
    return torch.bucketize(log_deg, bins) - 1

chem_bucket = create_bins(chem_deg)
dis_bucket  = create_bins(dis_deg)

pos_set = set((int(u), int(v)) for u,v in zip(train_pos[0],train_pos[1]))

def sample_negatives(num_samples):
    neg_edges = []

    chem_groups = {}
    dis_groups = {}

    for i in range(len(chem_bucket)):
        chem_groups.setdefault(int(chem_bucket[i]), []).append(i)
    for i in range(len(dis_bucket)):
        dis_groups.setdefault(int(dis_bucket[i]), []).append(i)

    while len(neg_edges) < num_samples:
        idx = random.randint(0, train_pos.size(1)-1)

        u = int(train_pos[0,idx])
        v = int(train_pos[1,idx])

        u_bucket = int(chem_bucket[u])
        v_bucket = int(dis_bucket[v])

        u_neg = random.choice(chem_groups[u_bucket])
        v_neg = random.choice(dis_groups[v_bucket])

        if (u_neg, v_neg) not in pos_set:
            neg_edges.append([u_neg, v_neg])

    return torch.tensor(neg_edges).t().contiguous()


In [5]:
train_neg = sample_negatives(train_pos.size(1))

train_edge_label_index = torch.cat([train_pos, train_neg], dim=1)
train_edge_label = torch.cat([
    torch.ones(train_pos.size(1)),
    torch.zeros(train_pos.size(1))
])

data[edge_type].edge_label_index = train_edge_label_index
data[edge_type].edge_label = train_edge_label


In [6]:
train_loader = LinkNeighborLoader(
    data,
    num_neighbors=[15, 10],
    batch_size=4096,
    edge_label_index=(edge_type, train_edge_label_index),
    edge_label=train_edge_label,
    shuffle=True,
)
def build_eval_loader(pos_edges):
    neg_edges = sample_negatives(pos_edges.size(1))

    edge_label_index = torch.cat([pos_edges, neg_edges], dim=1)
    edge_label = torch.cat([
        torch.ones(pos_edges.size(1)),
        torch.zeros(pos_edges.size(1))
    ])

    return LinkNeighborLoader(
        data,
        num_neighbors=[15, 10],
        batch_size=4096,
        edge_label_index=(edge_type, edge_label_index),
        edge_label=edge_label,
        shuffle=False,
    )

val_loader  = build_eval_loader(val_pos)
test_loader = build_eval_loader(test_pos)


In [ ]:
from torch_geometric.nn import to_hetero
from torch_geometric.nn import SAGEConv
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GNN(torch.nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_dim)
        self.conv2 = SAGEConv((-1, -1), hidden_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x
model = GNN(hidden_dim=128)
model = to_hetero(model, data.metadata(), aggr='sum')
model = model.to(device)
def decode(z_dict, edge_label_index):
    src, dst = edge_label_index
    return (z_dict['chemical'][src] *
            z_dict['disease'][dst]).sum(dim=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [19]:
from tqdm import tqdm
def train():
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        batch = batch.to(device)

        optimizer.zero_grad()

        z = model(batch.x_dict, batch.edge_index_dict)

        pred = decode(z, batch[edge_type].edge_label_index)

        loss = F.binary_cross_entropy_with_logits(
            pred,
            batch[edge_type].edge_label
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


In [23]:
def evaluate(loader):
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for batch in tqdm(loader):
            batch = batch.to(device)

            z = model(batch.x_dict, batch.edge_index_dict)
            pred = decode(z, batch[edge_type].edge_label_index)

            preds.append(pred.cpu())
            labels.append(batch[edge_type].edge_label.cpu())

    preds = torch.cat(preds)
    labels = torch.cat(labels)

    probs = preds.sigmoid().numpy()
    labels = labels.numpy()

    roc = roc_auc_score(labels, probs)
    ap  = average_precision_score(labels, probs)

    return roc, ap


In [24]:
print(data.metadata())


(['chemical', 'disease', 'gene'], [('chemical', 'chem_gene', 'gene'), ('chemical', 'chem_disease', 'disease'), ('gene', 'gene_disease', 'disease'), ('gene', 'rev_chem_gene', 'chemical'), ('disease', 'rev_chem_disease', 'chemical'), ('disease', 'rev_gene_disease', 'gene')])


In [ ]:
import os

os.makedirs("checks", exist_ok=True)


In [25]:
for epoch in range(1, 21):
    loss = train()
    val_roc, val_ap = evaluate(val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Loss {loss:.4f} | "
          f"Val ROC {val_roc:.4f} | "
          f"Val AP {val_ap:.4f}")
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_auc": val_roc
    }

    torch.save(checkpoint, f"checks/epoch_{epoch+1:03d}.pt")



100%|██████████| 143/143 [01:10<00:00,  2.03it/s]


Epoch 01 | Loss 0.8208 | Val ROC 0.5042 | Val AP 0.5086


RuntimeError: Parent directory checks does not exist.